# 02 features

Design decisions: 5 s windows with 2.5 s hop on a 5 Hz grid; gap-aware interpolation so sensor outages stay visible; simulator truth topics never read; receiver self-report columns kept in a separate `rx_*` family; onset and restore times taken from the log itself.

In [ ]:
# ============================================================
# BOOTSTRAP  (top of every notebook in this project)
# ============================================================
import os, sys, subprocess, shutil
from pathlib import Path

PROJECT   = "UAV_GNSS"
REPO_NAME = "uav-gnss-triage"

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT  = DRIVE_MOUNT / "MyDrive" / f"{PROJECT}_Research"
REPO_DIR    = DRIVE_ROOT / REPO_NAME

if not (DRIVE_MOUNT / "MyDrive").exists():
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))

for dotfile in (".gitconfig", ".git-credentials"):
    src = DRIVE_ROOT / dotfile
    if src.exists():
        shutil.copy(src, Path.home() / dotfile)
cred = Path.home() / ".git-credentials"
if cred.exists():
    os.chmod(cred, 0o600)
subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))
    import paths as P
print("CWD:", os.getcwd(), "| credentials:", cred.exists())


In [ ]:
RUN = "sih_flights_v2"
subprocess.run(["pip", "install", "-q", "pyulog", "pandas"], check=True)
out = P.FEATURES / RUN
proc = subprocess.Popen([sys.executable, str(P.SRC / "sih_features.py"), "--sih", str(P.SIH_RUNS / RUN),
                         "--whelan_zip", str(P.WHELAN_ZIP), "--out", str(out)],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
lines = [l for l in proc.stdout]
print("exit code:", proc.wait())
print("".join(l for l in lines if not l.startswith("f2026") and "FutureWarning" not in l and "W = pd.concat" not in l))
